In [0]:
from pyspark.sql import functions as F

orders_silver = spark.table("workspace.ecommerce_dataset.orders_silver")
customers_silver = spark.table("workspace.ecommerce_dataset.customers_silver")

trusted_order_count = orders_silver.count()
trusted_customer_count = customers_silver.count()

print(f"Trusted orders loaded: {trusted_order_count:,}")
print(f"Trusted customers loaded: {trusted_customer_count:,}")

orders_silver.createOrReplaceTempView("orders_silver")
customers_silver.createOrReplaceTempView("customers_silver")

In [0]:
completed_orders = orders_silver.filter(F.col("status") == "completed")
completed_count = completed_orders.count()

print(f"Total completed orders: {completed_count:,}")

In [0]:
total_revenue = completed_orders.agg(
    F.round(F.sum("total_amount"), 2).alias("total_revenue")
).first()["total_revenue"]

total_revenue = total_revenue or 0.0

print(f"Trusted revenue from completed orders: ${total_revenue:,.2f}")

In [0]:
revenue_by_city = (
    completed_orders
    .groupBy("city")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.count("*").alias("order_count")
    )
    .orderBy(F.desc("total_revenue"))
)

print("Revenue by City:")
display(revenue_by_city)

In [0]:
revenue_by_category = (
    completed_orders
    .groupBy("product_category")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.count("*").alias("order_count")
    )
    .orderBy(F.desc("total_revenue"))
)

print("Revenue by Product Category:")
display(revenue_by_category)

In [0]:
top_10_orders = (
    orders_silver
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "city",
        "product_category",
        "quantity",
        "unit_price",
        "total_amount",
        "status",
        "payment_method"
    )
    .orderBy(F.desc("total_amount"))
    .limit(10)
)

print("Top 10 Orders by Total Amount:")
display(top_10_orders)

In [0]:
completed_order_stats = completed_orders.agg(
    F.round(F.avg("total_amount"), 2).alias("average_order_value"),
    F.round(F.min("total_amount"), 2).alias("minimum_order_value"),
    F.round(F.max("total_amount"), 2).alias("maximum_order_value"),
    F.round(F.stddev("total_amount"), 2).alias("standard_deviation")
).first()

avg_order_value = completed_order_stats["average_order_value"] or 0.0

print(f"Average completed-order value: ${avg_order_value:,.2f}")
print(f"Minimum completed-order value: ${completed_order_stats['minimum_order_value']:,.2f}")
print(f"Maximum completed-order value: ${completed_order_stats['maximum_order_value']:,.2f}")
print(f"Standard deviation: ${completed_order_stats['standard_deviation']:,.2f}")

In [0]:
orders_by_status = orders_silver.groupBy("status").agg(
    F.count("*").alias("order_count")
).orderBy(F.desc("order_count"))

print("Orders by Normalized Status:")
display(orders_by_status)

for row in orders_by_status.collect():
    percentage = row["order_count"] / trusted_order_count * 100
    print(f"{row['status']}: {row['order_count']:,} ({percentage:.1f}%)")

In [0]:
payment_usage = orders_silver.groupBy("payment_method").agg(
    F.count("*").alias("usage_count")
).orderBy(F.desc("usage_count"), F.asc("payment_method"))

print("Payment Method Usage:")
display(payment_usage)

most_used_payment = payment_usage.first()

print(f"Most frequently used payment method: {most_used_payment['payment_method']}")
print(f"Used in {most_used_payment['usage_count']:,} orders")

In [0]:
top_5_cities = (
    completed_orders
    .groupBy("city")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.count("*").alias("order_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    )
    .orderBy(F.desc("total_revenue"))
    .limit(5)
)

print("Top 5 Cities by Trusted Revenue:")
display(top_5_cities)

top_city = top_5_cities.first()

print(f"Highest revenue city: {top_city['city']}")
print(f"Revenue: ${top_city['total_revenue']:,.2f}")

In [0]:
customer_context = (
    customers_silver
    .select(
        "customer_id",
        "customer_name",
        "customer_type",
        F.col("city").alias("customer_city")
    )
    .withColumn("_customer_matched", F.lit(True))
)

orders_enriched = (
    orders_silver
    .join(customer_context, on="customer_id", how="left")
    .withColumn("customer_matched", F.coalesce(F.col("_customer_matched"), F.lit(False)))
    .drop("_customer_matched")
)

orders_enriched.createOrReplaceTempView("orders_enriched")

print(f"Enriched orders: {orders_enriched.count():,}")
display(orders_enriched.limit(20))

In [0]:
orders_without_customer = orders_enriched.filter(~F.col("customer_matched"))
orphan_order_count = orders_without_customer.count()

print(f"Trusted orders without matching trusted customer: {orphan_order_count:,}")

if orphan_order_count > 0:
    display(orders_without_customer.limit(20))

In [0]:
completed_enriched = orders_enriched.filter(F.col("status") == "completed")

revenue_by_customer = (
    completed_enriched
    .filter(F.col("customer_name").isNotNull())
    .groupBy("customer_id", "customer_name", "customer_type")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.count("*").alias("order_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    )
    .orderBy(F.desc("total_revenue"))
)

print("Top Customers by Trusted Revenue:")
display(revenue_by_customer.limit(20))

top_customer = revenue_by_customer.first()

if top_customer:
    print(f"Top customer: {top_customer['customer_name']}")
    print(f"Revenue: ${top_customer['total_revenue']:,.2f}")

In [0]:
revenue_by_customer_type = (
    completed_enriched
    .filter(F.col("customer_type").isNotNull())
    .groupBy("customer_type")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.count("*").alias("order_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    )
    .orderBy(F.desc("total_revenue"))
)

print("Revenue by Customer Type:")
display(revenue_by_customer_type)

In [0]:
orders_classified = orders_silver.withColumn(
    "order_size",
    F.when(F.col("total_amount") < 50, "small")
    .when((F.col("total_amount") >= 50) & (F.col("total_amount") < 200), "medium")
    .when((F.col("total_amount") >= 200) & (F.col("total_amount") < 500), "large")
    .otherwise("premium")
)

classification_counts = (
    orders_classified
    .groupBy("order_size")
    .agg(F.count("*").alias("order_count"))
    .orderBy(
        F.when(F.col("order_size") == "small", 1)
        .when(F.col("order_size") == "medium", 2)
        .when(F.col("order_size") == "large", 3)
        .otherwise(4)
    )
)

print("Order Value Classification:")
display(classification_counts)

print("small   = below $50")
print("medium  = $50 to $199.99")
print("large   = $200 to $499.99")
print("premium = $500 or more")

In [0]:
boundary_check = spark.createDataFrame(
    [(49.99,), (50.00,), (199.99,), (200.00,), (499.99,), (500.00,)],
    ["total_amount"]
)

boundary_check = boundary_check.withColumn(
    "order_size",
    F.when(F.col("total_amount") < 50, "small")
    .when((F.col("total_amount") >= 50) & (F.col("total_amount") < 200), "medium")
    .when((F.col("total_amount") >= 200) & (F.col("total_amount") < 500), "large")
    .otherwise("premium")
)

print("Classification boundary check:")
display(boundary_check)

In [0]:
sql_completed_count = spark.sql("""
    SELECT COUNT(*) AS completed_orders
    FROM orders_silver
    WHERE status = 'completed'
""").first()["completed_orders"]

print(f"DataFrame API: {completed_count:,}")
print(f"Spark SQL:     {sql_completed_count:,}")
print(f"Match: {completed_count == sql_completed_count}")

In [0]:
sql_total_revenue = spark.sql("""
    SELECT ROUND(SUM(total_amount), 2) AS total_revenue
    FROM orders_silver
    WHERE status = 'completed'
""").first()["total_revenue"]

sql_total_revenue = sql_total_revenue or 0.0

print(f"DataFrame API: ${total_revenue:,.2f}")
print(f"Spark SQL:     ${sql_total_revenue:,.2f}")
print(f"Match: {total_revenue == sql_total_revenue}")

In [0]:
sql_payment_method = spark.sql("""
    SELECT payment_method, COUNT(*) AS usage_count
    FROM orders_silver
    GROUP BY payment_method
    ORDER BY usage_count DESC, payment_method ASC
    LIMIT 1
""").first()

print(f"DataFrame API: {most_used_payment['payment_method']} ({most_used_payment['usage_count']:,})")
print(f"Spark SQL: {sql_payment_method['payment_method']} ({sql_payment_method['usage_count']:,})")

payment_method_match = (
    most_used_payment["payment_method"] == sql_payment_method["payment_method"] and
    most_used_payment["usage_count"] == sql_payment_method["usage_count"]
)

print(f"Match: {payment_method_match}")

In [0]:
orders_raw = spark.read.csv(
    "/Volumes/workspace/ecommerce_dataset/ecommerce/orders_raw.csv",
    header=True,
    inferSchema=True
)

customers_raw = spark.read.csv(
    "/Volumes/workspace/ecommerce_dataset/ecommerce/customers_raw.csv",
    header=True,
    inferSchema=True
)

raw_order_count = orders_raw.count()
raw_customer_count = customers_raw.count()

excluded_order_count = raw_order_count - trusted_order_count
excluded_customer_count = raw_customer_count - trusted_customer_count

print("DATA QUALITY IMPACT")

print("\nOrders:")
print(f"Raw: {raw_order_count:,}")
print(f"Trusted: {trusted_order_count:,}")
print(f"Excluded: {excluded_order_count:,}")
print(f"Success rate: {trusted_order_count/raw_order_count*100:.2f}%")

print("\nCustomers:")
print(f"Raw: {raw_customer_count:,}")
print(f"Trusted: {trusted_customer_count:,}")
print(f"Excluded: {excluded_customer_count:,}")
print(f"Success rate: {trusted_customer_count/raw_customer_count*100:.2f}%")

print(f"\nTrusted orders without a matching customer: {orphan_order_count:,}")

In [0]:
top_category = revenue_by_category.first()

print(f"Completed orders available for reporting: {completed_count:,}")
print(f"Trusted completed-order revenue: ${total_revenue:,.2f}")
print(f"Average completed-order value: ${avg_order_value:,.2f}")

if top_city:
    print(f"Strongest city by revenue: {top_city['city']} (${top_city['total_revenue']:,.2f})")

if top_category:
    print(f"Strongest product category: {top_category['product_category']} (${top_category['total_revenue']:,.2f})")

print(f"Most frequently used payment method: {most_used_payment['payment_method']}")
print(f"Order cleaning success rate: {trusted_order_count/raw_order_count*100:.2f}%")
print(f"Trusted orders without customer enrichment: {orphan_order_count:,}")